C:\Users\gtamo\Serac Biosciences Dropbox\Serac_team\12_Proteomics\5_Inventory\CDDVault\Proteomics\GiorgioTamo

In [ ]:
%cd ../.
import os, sys
sys.path.insert(0, os.path.abspath('../Scripts'))


In [ ]:
%load_ext autoreload
%autoreload 2

import re
import os
import pandas as pd
import numpy as np
import py3Dmol
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import csv

from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem,rdFMCS
# import prolif as plf
from glob import glob
import meeko
import subprocess as sub
# from vina import Vina
import time
from tqdm import tqdm
tqdm.pandas()
import importlib
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.dummy import DummyClassifier
from sklearn.manifold import TSNE
from xgboost import XGBClassifier,XGBRegressor
from openTSNE import TSNE as oTSNE        # pip install openTSNE
import seaborn as sns
from scipy import stats

# user defined modules
import Rdkit_tools as rdkit_tools
importlib.reload(rdkit_tools)
import Molecule as M
import ML_Reg as ML_Reg
import ML_Class as ML_Class
from MolViz3D import MolViz3D
import Statistics_tools as stats_tools
import python.functions as fn
# from tdc.multi_pred import DTI

In [ ]:
## params
active_c = '#008bfb' # "#3B85C1"
silent_c = '#ff0051'

RAW_PROTEOMICS_PATH   = 'data/MS/20260424_Proteomics_Database_CSV_Export.csv'
CLEAN_PROTEOMICS_PATH = 'data/MS/CDD CSV Export - 2026-04-29 06h13m33s.csv'
CHEMLIB_PATH          = 'data/chemical_libs/20260430_SERAC_lib.csv' # smiles + compound
OT_ROOT               = 'data/external/opentarget'
OT_CACHE              = 'output/MS/opentargets_target_disease.parquet'
GENE_SAR_OUT          = 'output/MS/20260509_geneSAR_R2_full_genome.csv' # R2 per gene
MCS_CSV               = 'output/MS/20260505_target_final_mcs.csv' # MCS enrichment
PATENTS               = 'data/patent/20260511_pharma_sm.csv' # pharma targets of interest

## 0. Imports

In [ ]:
%%time
## raw proteomics data
df_raw   = pd.read_csv(RAW_PROTEOMICS_PATH)
## compounds ids and smiles
serac_df = (pd.read_csv(CHEMLIB_PATH)
            .drop(['CDD Number','Synonyms'],axis=1)
            .rename(columns={'Molecule Name':'compound','SMILES':'smiles','Projects':'project'})
            .drop('project',axis=1)
            .drop_duplicates())

## Get clean MS data to get plate Ids
MS = pd.read_csv(CLEAN_PROTEOMICS_PATH).drop(['CDD Number'],axis=1)
MS = MS[MS['MSData - Proteomics activities: Source']=='SERAC'] # 
MS['MSData - Proteomics activities: Date'] = pd.to_datetime(MS['MSData - Proteomics activities: Date'])
MS['MSData - Proteomics activities: MSPlate'].unique()

# remove plate 12 data (from Daniela)
MS = MS[~MS['MSData - Proteomics activities: MSPlate'].isin(['Pw33','Plate12']) ]

# if a compound is tested multiple times, get only by latest date
MS = MS.sort_values('MSData - Proteomics activities: Date',ascending=False).reset_index()
MS = MS.groupby('Molecule Name').first().reset_index()

print('>', len(MS['Molecule Name'].unique()),'unique compounds')
print('> Ligase(s)',list(MS['MSData - Proteomics activities: Ligase'].unique()))
print('> Cellline',list(MS['MSData - Proteomics activities: Cell line'].unique()))
print('> dim:',MS.shape)
MS.head(2)


In [ ]:
%%time
## Clean, filter and format raw MS data
df_raw = df_raw[~df_raw['MSPlate'].isin(['Plate12','Plate15','Plate23']) ]
df_raw = df_raw[df_raw['MoleculeBatchID'].isin(MS['MSData - Proteomics activities: Molecule-Batch ID'])]
# df_raw['compound'] = df_raw['MoleculeBatchID'].progress_apply(lambda x: '-'.join(x.split('-')[:2]))
# df_raw['batch'] = df_raw['MoleculeBatchID'].progress_apply(lambda x: '-'.join(x.split('-')[2:]))
parts = df_raw['MoleculeBatchID'].str.split('-', n=2, expand=True)
df_raw['compound'] = parts[0] + '-' + parts[1]   # 'SRB-0000385'
df_raw['batch']    = parts[2]                    # '001'
df_raw.head(2)

In [ ]:
%%time
## Get smiles from library and compute MF features
serac_df = serac_df.dropna().reset_index(drop=True)
failed_CMs = rdkit_tools.check_smiles_RDKiT(serac_df)
serac_df = serac_df[~serac_df['compound'].isin(serac_df)]
## then compute features:
if 11<3:
    properties  = rdkit_tools.compute_properties_from_smiles(serac_df)
    MF          = rdkit_tools.get_MF_bits_from_df(serac_df,nBits=2048)
    MF_features = pd.merge(MF,properties)
else:
    MF_features = rdkit_tools.compute_H236_features(serac_df, v=True)
MF_features.head(1)

In [ ]:
## OpenTargets disease scores for every gene seen in the proteomics data.
## Cached to parquet — re-run is instant. Delete the file (or change OT_CACHE)
## to force a recompute from the local bulk dump under data/external/opentarget/.

if os.path.exists(OT_CACHE):
    print(f'> loading cached {OT_CACHE}')
    ot_df = pd.read_parquet(OT_CACHE)
else:
    # normalise the gene column: explode multi-gene strings (peptides mapping to >1
    # protein), strip whitespace, drop empties / NaNs, dedupe.
    unique_genes = pd.DataFrame({
        'gene': (df_raw['genes'].dropna().astype(str)
                  .str.split(r'[;,|]', regex=True).explode()
                  .str.strip()
                  .replace('', pd.NA).dropna()
                  .unique())
    })
    print(f'> {len(unique_genes):,} unique gene symbols in df_raw')

    ot_df = fn.get_opentarget_disease_score(
        unique_genes, gene_col='gene',
        top_n=20,
        ot_root=OT_ROOT,
    )
    os.makedirs(os.path.dirname(OT_CACHE), exist_ok=True)
    ot_df.to_parquet(OT_CACHE, index=False)
    print(f'> wrote {OT_CACHE}')

print(f'> {len(ot_df):,} (target, disease) rows  /  '
      f'{ot_df["target_symbol"].nunique():,} targets resolved')
ot_df.head(2)

In [ ]:
## disease areas of interest to big pharma:
## Filter ot_df to (target, disease) rows whose `therapeutic_areas` overlap with
## big-pharma R&D priority franchises. The default token set is BMS-style:
## oncology + hematology + immunology + cardio + neuro + psych + metabolic +
## endocrine. Tweak PRIORITY for a different pharma profile (e.g. Lilly leans
## metabolic/CNS, Roche leans onco/CNS/ophtho, Pfizer leans vaccines/inflammation).
## DROP catches OT placeholder labels (phenotype, measurement, biological_process)
## and out-of-scope categories (animal disease, medical procedure) so they don't
## let a row slip through on their own.

PRIORITY = {
    # Annotations: flagship products per pharma, just to anchor priority intuition.
    'cancer or benign tumor',                       # universal — BMS Opdivo/Yervoy, Roche Herceptin/Tecentriq/Phesgo, NVS Kisqali/Kymriah/Pluvicto, Pfizer Ibrance/Padcev, Lilly Verzenio/Jaypirca
    'hematologic disease',                          # BMS Revlimid/Pomalyst (multiple myeloma), Pfizer Elrexfio, NVS Tasigna, Roche Hemlibra (haemophilia), Lilly Jaypirca
    'cardiovascular disease',                       # BMS Eliquis, NVS Entresto/Leqvio (PCSK9 siRNA), Pfizer Vyndaqel (TTR amyloidosis)
    'immune system disease',                        # BMS Zeposia/Sotyktu/Orencia, NVS Cosentyx/Xolair, Pfizer Xeljanz/Cibinqo, Lilly Olumiant/Taltz, Roche Actemra
    'musculoskeletal or connective tissue disease', # RA / lupus / psoriasis (overlaps immune) — BMS Orencia, NVS Cosentyx, Pfizer Xeljanz, Lilly Olumiant
    'nervous system disease',                       # NVS Kesimpta/Gilenya (MS), Roche Ocrevus/Evrysdi (MS, SMA), Lilly Kisunla (AD, donanemab), BMS Cobenfy (post-Karuna)
    'psychiatric disorder',                         # BMS Cobenfy (schizophrenia), Lilly historic SSRIs/atypicals, otherwise smaller pharma footprint
    'nutritional or metabolic disease',             # Lilly #1 (Mounjaro / Zepbound — obesity/T2D), Pfizer & NVS chasing GLP-1 follow-ons
    'endocrine system disease',                     # Lilly (hormone/metabolic overlap), oncology adjacency (hormone-driven cancers) for BMS/Roche
    'disorder of visual system',                  # ← uncomment for Roche specifically (Vabysmo/Lucentis ophthalmology franchise)
    'respiratory or thoracic disease',            # ← uncomment for COPD/asthma plays (NVS Xolair, GSK; not named-four core)
    'infectious disease',                         # ← uncomment for vaccines/antivirals (Pfizer Comirnaty/Paxlovid/Prevnar)
}
DROP = {'phenotype', 'measurement', 'biological_process',
        'animal disease', 'medical procedure'}

def is_pharma_relevant(ta_str):
    """Keep rows whose therapeutic_areas overlap PRIORITY and aren't pure DROP-only."""
    if not isinstance(ta_str, str) or not ta_str:
        return False
    tokens = set(ta_str.split('|'))
    return bool(tokens & PRIORITY) and bool(tokens - DROP)

ot_pharma = ot_df[ot_df['therapeutic_areas'].apply(is_pharma_relevant)].reset_index(drop=True)
print(f'> {len(ot_pharma):,} / {len(ot_df):,} (target, disease) rows pass the BMS-style filter')
print(f'> covers {ot_pharma["target_symbol"].nunique():,} unique targets')

# breakdown by which priority area each row hits (a row can hit multiple)
print('\nrows per priority area:')
for token in sorted(PRIORITY):
    n = ot_pharma['therapeutic_areas'].str.split('|').apply(lambda s: token in s).sum()
    print(f'  {n:>6,}  {token}')

# stricter alternatives if you want to scope tighter:
# ot_oncology_hem = ot_df[ot_df['therapeutic_areas'].apply(
#     lambda s: bool({'cancer or benign tumor', 'hematologic disease'} & set(s.split('|')))
#               if isinstance(s, str) else False
# )]
# ot_pure_onco = ot_df[ot_df['therapeutic_areas'] == 'cancer or benign tumor']

ot_pharma.head(2)

## 1. Testing 1 gene at time

In [ ]:
run_df = pd.read_csv('output/MS/20260504_0.8_MS.csv')
run_df.sort_values('R2',ascending=False).head(20)

In [ ]:
gene = 'KDM1B' # 'TUBB8' #  # 'ASNS' #  'TP53'  # 'UNC45A' # 'PLIN2' # 'TMEM205' #      
tardf = df_raw[df_raw['genes']==gene]
tardf.shape

In [ ]:
# tardf['compound'] = tardf['MoleculeBatchID'].apply(lambda x: '-'.join(x.split('-')[:2]))
# tardf['batch'] = tardf['MoleculeBatchID'].apply(lambda x: '-'.join(x.split('-')[2:]))
# len(tardf['compound'].unique())

In [ ]:
tardf.groupby('batch').size()

In [ ]:
tardf_ml       = tardf.groupby('compound')['logfc'].mean().reset_index()
tardf_ml['label'] = tardf_ml['logfc']
# binarize dataset?
# tardf_ml['label'] = (tardf_ml['logfc'] < -1.0) * 1

tardf_ml.head(3)

In [ ]:
data = pd.merge(serac_df,tardf_ml).drop_duplicates()
print(data.shape)
data.head(3)

In [ ]:
%%time
# properties = rdkit_tools.compute_properties_from_smiles(data)
# MF         = rdkit_tools.get_MF_bits_from_df(data,nBits=2048)
# ML_data = pd.merge(MF,properties)
ML_data = pd.merge(MF_features, tardf_ml[['compound','label']], on='compound').dropna()
print(ML_data.shape)
ML_data.head(2)

In [ ]:
## RandomForestRegressor grid search — tight grid on the 4 most impactful knobs.
## Skipped on purpose (compute grows fast, returns flatten): bootstrap, criterion,
## min_samples_split, ccp_alpha. Bring them back later only if the leaderboard plateaus.
##   - n_estimators: more trees = lower variance, plateaus past a few hundred for n~1.6k
##   - max_depth: None (unlimited) is RF's default; cap to test if shallower trees help
##   - max_features: signature RF regulariser; controls per-split decorrelation
##   - min_samples_leaf: cheap second-line regulariser
## Grid size: 2 × 3 × 2 × 2 = 24 combos × 5 CV folds = 120 fits  (~5–15 min on 10 cores)
## Scoring: R² — the harness's default for regression.
if 11<3:
    rf_params = {
        'n_estimators':     [100, 200],
        'max_depth':        [None, 10, 20],
        'max_features':     [0.3, 'sqrt'],
        'min_samples_leaf': [1, 5],
        # leave defaults: bootstrap=True, criterion='squared_error', min_samples_split=2
    }

    res_search_rf = ML_Reg.Model_gridsearch_parpool(
        ML_data.drop(['compound','smiles'], axis=1),
        'RandomForestRegressor', rf_params,
        col2scale_=[], nprocs=10, multiPtype='joblib',
    )
    res_search_rf.head(10)

In [ ]:
## XGBRegressor grid search — tight grid on the 4 most impactful knobs.
## Skipped on purpose (compute drops fast, returns don't): subsample, colsample,
## gamma, reg_alpha, reg_lambda. Bring them back later only if the leaderboard plateaus.
##   - n_estimators / learning_rate: trade trees for step size; coupled, sweep both
##   - max_depth: most impactful single knob; > 7 typically overfits with n~1.6k
##   - min_child_weight: cheap regularisation
## Grid size: 2 × 3 × 2 × 2 = 24 combos × 3 CV folds = 72 fits  (~5–10 min on 10 cores)
## Scoring: R² — the harness's default for regression.
if 11 < 3: 
    xgb_params = {
        'n_estimators':     [100,200],
        'max_depth':        [3, 5, 7],
        'learning_rate':    [0.05, 0.1],
        'min_child_weight': [1, 5],
        # leave defaults: subsample=1, colsample_bytree=1, gamma=0, reg_alpha=0, reg_lambda=1
    }

    res_search = ML_Reg.Model_gridsearch_parpool(
        ML_data.drop(['compound','smiles'], axis=1),
        'XGBRegressor', xgb_params,
        col2scale_=[], nprocs=10, multiPtype='joblib',
    )
    res_search.head(10)

In [ ]:
def null_r2(ML_data,model, n_perm=5):
    nulls = []
    for seed in range(n_perm):
        ML_perm = ML_data.copy()
        ML_perm['label'] = ML_perm['label'].sample(frac=1, random_state=seed).values
        trained_model, df_pred =  ML_Reg.run_K_Fold_Xval_Regression(ML_perm, model=model,col_to_rm=['compound','label'],
                                         ID='compound', get_ints=False, v=False,to_impute=None,rm_empty_cols=False)
        nulls.append(ML_Reg.get_reg_metrics_from_preddf(df_pred, v=False)['r2'])
    return np.array(nulls)

model = RandomForestRegressor(**params_rf, n_jobs=-1, random_state=0)
nr2 = null_r2(ML_data,model, n_perm=5).mean()


In [ ]:
%%time
# params_xgb = {'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.05, 'colsample_bytree': 1.0, 'min_child_weight': 1}
params_rf = {'n_estimators':100,'max_depth':20,'max_features':'sqrt','min_samples_leaf':1}
# model = XGBRegressor(**params_xgb, n_jobs=-1, random_state=0)
model = RandomForestRegressor(**params_rf, n_jobs=8, random_state=0)
trained_model, df_pred =  ML_Reg.run_K_Fold_Xval_Regression(ML_data, model=model,col_to_rm=['compound','label'], ID='compound', get_ints=False, v=True,to_impute=None,rm_empty_cols=False)

In [ ]:
# model = RandomForestClassifier(n_estimators=100,n_jobs=-1,random_state=0) # 
ML_data['label'] = (ML_data['label'] < -1.0) * 1
stats_tools.check_ML_data(ML_data)

params = {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 1.0, 'min_child_weight': 1, 'scale_pos_weight': 1}
model = XGBClassifier(**params,n_jobs=-1,random_state=42)
trained_model, df_pred =  ML_Class.run_K_Fold_Xval_Classification(ML_data, ID='compound', model=model,
                                   folds=5, col_to_rm=['compound','label'], v=True, ctf=0.5, impute_by_mean=False)


In [ ]:
roc_df = ML_Class.plot_roc_curve(
    [df_pred],
    c=[active_c],
    l=['XGB'],
    metric2show=['roc_auc'],dpi=70,legend=True
)

## 2. Compute R2 for every gene
Run K-fold X-validation for each gene of the ot list

In [ ]:
## get target list
ot_df_rank = (ot_pharma.sort_values('overall_score',ascending=False).reset_index(drop=True).groupby('target_symbol').first().sort_values('overall_score',ascending=False).reset_index())
target_list = list(ot_df_rank[ot_df_rank['overall_score'] > 0.4]['target_symbol'])

## add pharma target list:
patents = list(pd.read_csv('data/patent/20260511_pharma_sm.csv')['gene'].unique())
target_list += patents

target_list = list(dict.fromkeys(target_list))
len(target_list)


In [ ]:
## per-gene SAR screen — RESUMABLE.
## Each completed gene is appended to the CSV and flushed immediately, so a crash
## (or kernel kill) loses at most one in-flight gene. Re-running the cell skips
## any gene already present in the output file.
## Per-gene CV is delegated to fn.compute_gene_sar_r2 (python/functions.py).

# H236 production-model params (from autoresearch deployable champion):
#   single RF + multi-FP features + winsorize labels + drop [Plate12,15,23].
#   matches python/compute_R2_for_all_genes.py and the iter-83 deployable champion.
PARAMS_RF = {'n_estimators': 200, 'max_depth': 20,
             'max_features': 0.3, 'min_samples_leaf': 2,
             'min_samples_split': 4}

# Per-gene label override (matches python/compute_R2_for_all_genes.yaml):
#   default → per-gene 1-99% winsorize ('logfc_clipped')
#   override → raw 'logfc' for genes where winsorize was found to hurt R²
RAW_LOGFC_GENES = {
    'PARP4',    # 0.194 → 0.261 (Δ=-0.067 with winsorize, biggest)
    'TGFBR3',   # 0.095 → 0.129
    'MERTK',    # 0.172 → 0.193
    'PIK3CA',   # 0.019 → 0.039
    'MCL1',     # 0.027 → 0.037
    'PIK3CD',   # 0.020 → 0.030
    'MDM2',     # 0.060 → 0.068
    'ROCK1',    # 0.050 → 0.058
    'UNC45A',   # 0.111 → 0.135 (calibration cohort)
}

# Pre-compute per-gene winsorized labels once — added as a new column.
print('> computing logfc_clipped (per-gene 1-99% winsorize)...')
df_raw['logfc_clipped'] = df_raw['logfc'].copy()
for _g, _idx in df_raw.groupby('genes').groups.items():
    _vals = df_raw.loc[_idx, 'logfc'].values
    _mask = ~pd.isna(_vals)
    if _mask.sum() < 20:
        continue
    _lo, _hi = np.nanpercentile(_vals[_mask], [1, 99])
    df_raw.loc[_idx, 'logfc_clipped'] = np.clip(_vals, _lo, _hi)

os.makedirs(os.path.dirname(GENE_SAR_OUT), exist_ok=True)

# Resume: read which genes are already done from the checkpoint CSV
done = set()
if os.path.exists(GENE_SAR_OUT) and os.path.getsize(GENE_SAR_OUT) > 0:
    done = set(pd.read_csv(GENE_SAR_OUT)['gene'])
    print(f'> resuming: {len(done):,} genes already in {GENE_SAR_OUT}')

todo = [g for g in target_list if g not in done]
print(f'> {len(todo):,} / {len(target_list):,} genes to process')
print(f'> raw-logfc override list ({len(RAW_LOGFC_GENES)}): '
      f'{sorted(RAW_LOGFC_GENES)}')

# Append mode + write header only if the file is new/empty
write_header = (not os.path.exists(GENE_SAR_OUT)) or os.path.getsize(GENE_SAR_OUT) == 0
with open(GENE_SAR_OUT, 'a', newline='') as fh:
    writer = csv.DictWriter(fh, fieldnames=['gene', 'R2', 'nullR2', 'n'])
    if write_header:
        writer.writeheader()

    for gene in tqdm(todo):
        # Per-gene label: default winsorize, override → raw for select targets
        label_col = 'logfc' if gene in RAW_LOGFC_GENES else 'logfc_clipped'
        result = fn.compute_gene_sar_r2(
            gene, df_raw, MF_features,
            label_col=label_col,
            model_class=RandomForestRegressor,
            model_params=PARAMS_RF,
            min_compounds=100,
            n_jobs=8, seed=0,
            ML_Reg_module=ML_Reg,
        )

        # checkpoint after each gene — disk-safe past this point
        writer.writerow(result)
        fh.flush()

        if pd.notna(result['R2']) and result['R2'] > 0.1:
            tqdm.write(f"> {result['gene']:<12}  R²={result['R2']:.3f}  "
                       f"null={result['nullR2']}  n={result['n']}  "
                       f"label={label_col}")


In [ ]:
# load the (now-complete) output for downstream use
target2R2_df = pd.read_csv(GENE_SAR_OUT)
print(f'> {len(target2R2_df):,} rows in {GENE_SAR_OUT}')

# annotate each gene with how many compounds dropped below the activity threshold
# (= a quick proxy for "how many strong actives" the gene actually has)
LOGFC_THRESHOLD = -1.0
mean_per_compound = df_raw.groupby(['genes', 'compound'])['logfc'].mean().reset_index()
no_below = (mean_per_compound[mean_per_compound['logfc'] < LOGFC_THRESHOLD]
            .groupby('genes').size()
            .rename('no_cmp_below_ctf'))
target2R2_df = (target2R2_df
                .merge(no_below, left_on='gene', right_index=True, how='left')
                .fillna({'no_cmp_below_ctf': 0})
                .astype({'no_cmp_below_ctf': int}))

## 3. Enrichment analysis

In [ ]:
target_final = target2R2_df[target2R2_df['n'] > 400].sort_values('R2', ascending=False)
target_final.head(20)

In [ ]:
## Per-gene MCS enrichment for the shortlist `target_final` — RESUMABLE + PARALLEL.
##
## Three speedups vs the naive serial loop:
##   1) **Pre-extract** per-gene (smiles, logfc) into a dict so workers don't pickle df_raw.
##   2) **Threading** (not loky) — RDKit's rdFMCS is C++ and releases the GIL,
##      same logic as the SAR-screen cell that benefited from threading.
##   3) **Incremental CSV checkpoint** — each finished gene is appended + flushed,
##      so a crash mid-run loses one gene of work, not the whole batch.
## Expected: 2h → ~15–20 min on N_JOBS=8 threads.
import csv, contextlib, threading, joblib
from joblib import Parallel, delayed

N_JOBS    = 8

# tqdm-on-joblib helper (per-task, not per-dispatch)
@contextlib.contextmanager
def _tqdm_joblib(pbar):
    class _Cb(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *a, **kw):
            pbar.update(n=self.batch_size)
            return super().__call__(*a, **kw)
    prev = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = _Cb
    try:
        yield pbar
    finally:
        joblib.parallel.BatchCompletionCallBack = prev
        pbar.close()

# 1) Read the cache FIRST so we can skip the pre-extraction entirely if everything's done.
os.makedirs(os.path.dirname(MCS_CSV), exist_ok=True)
done = set()
if os.path.exists(MCS_CSV) and os.path.getsize(MCS_CSV) > 0:
    done = set(pd.read_csv(MCS_CSV)['gene'])
    print(f'> cache: {len(done):,} genes already in {MCS_CSV}')

remaining = [g for g in target_final['gene'] if g not in done]
print(f'> remaining: {len(remaining):,} / {len(target_final):,} genes need processing')

# 2) Pre-extract per-gene compound list — ONLY for the remaining genes
gene_data = {}
if remaining:
    serac_smi = serac_df.set_index('compound')['smiles']
    for gene in tqdm(remaining, desc='preflight'):
        sub = df_raw[df_raw['genes'] == gene]
        y = sub.groupby('compound')['logfc'].mean().dropna()
        cmpds = y.index.intersection(serac_smi.index)
        if len(cmpds) < 50:
            continue
        gene_data[gene] = pd.DataFrame({
            'smiles': serac_smi.loc[cmpds].values,
            'logfc':  y.loc[cmpds].values,
        })
    print(f'> {len(gene_data):,} of those pass preflight (≥50 compounds)')

todo = gene_data    # already filtered to undone-only above

# 3) Worker — only sees the small per-gene df, no closure over df_raw
def eval_one(gene, df_g):
    try:
        res = rdkit_tools.compute_MCS_enrichment(
            df_g, smiles_col='smiles', score_col='logfc',
            K_values=(5, 10, 20),
            mcs_threshold=0.7, mcs_timeout=15,
            rank_by='fisher_p',
        )
    except Exception:
        res = {}
    if not res:
        return {'gene': gene}
    return {
        'gene':        gene,
        'K':           res['K'],
        'fold':        res['fold'] if res['fold'] != float('inf') else 9999.0,
        'fisher_p':    res['fisher_p'],
        'in_top_pct':  res['in_top_pct'],
        'in_rest_pct': res['in_rest_pct'],
        'mcs_atoms':   res['mcs_atoms'],
        'mcs_smarts':  res['mcs_smarts'],
    }

# 4) Run in parallel + write each finished gene incrementally
fields = ['gene', 'K', 'fold', 'fisher_p', 'in_top_pct', 'in_rest_pct',
          'mcs_atoms', 'mcs_smarts']
write_header = (not os.path.exists(MCS_CSV)) or os.path.getsize(MCS_CSV) == 0
write_lock = threading.Lock()

if todo:
    with open(MCS_CSV, 'a', newline='') as fh:
        writer = csv.DictWriter(fh, fieldnames=fields)
        if write_header:
            writer.writeheader()

        FOLD_HIT_THRESHOLD = 10.0    # print live hits whose MCS fold-enrichment exceeds this
        def _checkpoint(rec):
            row = {f: rec.get(f) for f in fields}
            with write_lock:
                writer.writerow(row)
                fh.flush()
                # surface promising hits as they finish, like the SAR screen does
                fold = rec.get('fold')
                if fold is not None and (fold == 9999.0 or fold >= FOLD_HIT_THRESHOLD):
                    fold_str = '∞' if fold == 9999.0 else f'{fold:.1f}'
                    tqdm.write(
                        f'> {rec["gene"]:<12}  K={rec["K"]:<2}  fold={fold_str:>5}×  '
                        f'p={rec["fisher_p"]:.1e}  in_top={rec["in_top_pct"]:>5.1f}%  '
                        f'in_rest={rec["in_rest_pct"]:>5.2f}%  mcs={rec["mcs_atoms"]} atoms'
                    )
            return rec

        with _tqdm_joblib(tqdm(total=len(todo), desc='MCS enrichment')):
            _ = Parallel(n_jobs=N_JOBS, backend='threading')(
                delayed(lambda g, d: _checkpoint(eval_one(g, d)))(g, d)
                for g, d in todo.items()
            )
    print(f'> wrote / appended results to {MCS_CSV}')

# 5) Load + merge into target_final
mcs_df = pd.read_csv(MCS_CSV)
# inf was stored as 9999.0 in CSV — restore for clarity in display
mcs_df['fold'] = mcs_df['fold'].replace(9999.0, float('inf'))
target_final = target_final.drop(columns=[c for c in fields[1:] if c in target_final.columns]).merge(
    mcs_df, on='gene', how='left',
)


# 6) Per-gene logfc extremes — most-positive (max_logfc) & most-negative (min_logfc)
#    across compounds. For degrader analysis min_logfc is the "best hit" magnitude;
#    max_logfc is a sanity check for upregulators. Computed once via groupby.
logfc_extremes = (df_raw.groupby(['genes', 'compound'])['logfc'].mean().reset_index()
                  .groupby('genes')['logfc']
                  .agg(min_logfc='min', max_logfc='max').reset_index()
                  .rename(columns={'genes': 'gene'}))
target_final = (target_final
                .drop(columns=[c for c in ['min_logfc', 'max_logfc'] if c in target_final.columns])
                .merge(logfc_extremes, on='gene', how='left'))

cols = ['gene', 'R2', 'n', 'no_cmp_below_ctf', 'min_logfc', 'max_logfc',
        'K', 'fold', 'fisher_p', 'in_top_pct', 'in_rest_pct', 'mcs_atoms']

## add disease relevance
target_final = pd.merge(target_final,ot_df_rank.rename(columns={'target_symbol':'gene'})[['gene','overall_score']])

target_final.head(3)

In [ ]:
# ## plot R2 vs disease relevance
# ## Scatter of model-predictability (R²) vs OpenTargets disease association
# ## (overall_score). The chemist-actionable quadrant is top-right: high R² +
# ## high disease relevance. Top-20 closest to that corner get labelled, plus
# ## any genes in MUST_INCLUDE you want surfaced regardless of position.
# ## Marker palette mirrors the ggplot-style soft-blue/steel-edge look.
# from adjustText import adjust_text

# MUST_INCLUDE = ['UNC45A', 'KDM1B']     # always label these on top of the top-20

# # matched palette (ColorBrewer "Blues") — soft fill + darker edge
# HL_FILL = '#9ecae1'
# HL_EDGE = '#3182bd'

# # 1) merge disease scores onto target_final (use the per-target max from ot_df_rank)
# plot_df = target_final.copy() ## DO NOT MODIFY
# print(f'> {len(plot_df):,} genes with both R² and overall_score')

# # 2) distance from each point to the top-right corner in min-max-normalised space
# xn = (plot_df['R2']            - plot_df['R2'].min()           ) / (plot_df['R2'].max()            - plot_df['R2'].min())
# yn = (plot_df['overall_score'] - plot_df['overall_score'].min()) / (plot_df['overall_score'].max() - plot_df['overall_score'].min())
# plot_df = plot_df.assign(_dist_topright=np.sqrt((1 - xn)**2 + (1 - yn)**2))

# top20 = plot_df.nsmallest(20, '_dist_topright')
# must  = plot_df[plot_df['gene'].isin(MUST_INCLUDE)]
# missing = [g for g in MUST_INCLUDE if g not in plot_df['gene'].values]
# if missing:
#     print(f'  [warn] MUST_INCLUDE not found in plot_df: {missing}')

# # union the two highlight sets (corner-driven + must-include)
# highlighted = pd.concat([top20, must]).drop_duplicates('gene')

# # 3) plot
# fig, ax = plt.subplots(figsize=(9, 7), dpi=120)
# ax.scatter(plot_df['R2'], plot_df['overall_score'],
#            c='lightgrey', s=22, alpha=1, edgecolors='none',
#            label=f'all ({len(plot_df):,})')
# ax.scatter(highlighted['R2'], highlighted['overall_score'],
#            facecolor=HL_FILL, s=80, alpha=0.95,
#            edgecolors=HL_EDGE, linewidths=1.2,
#            label=f'highlighted ({len(highlighted)})',
#            zorder=5)

# # 4) labels with adjust_text so they don't overlap
# texts = [ax.text(r['R2'], r['overall_score'], r['gene'],
#                  fontsize=9, fontweight='bold')
#          for _, r in highlighted.iterrows()]
# adjust_text(texts,
#             ax=ax,
#             arrowprops=dict(arrowstyle='-', color='grey', alpha=0.5, lw=0.7),
#             expand_points=(1.3, 1.3))

# ax.set_xlabel('R² (5-fold CV) — predictability of logfc from structure')
# ax.set_ylabel('OpenTargets overall_score — disease association')
# ax.set_title('SAR predictability × disease relevance')
# ax.grid(alpha=0.3)
# ax.legend(loc='lower left', frameon=False, fontsize=9)
# ax.set_axisbelow(True)
# plt.tight_layout()
# plt.show()

# # table of the highlighted set, sorted by (R², overall_score)
# highlighted[['gene', 'R2', 'overall_score', 'no_cmp_below_ctf', 'fold', 'fisher_p']].sort_values(
#     ['R2', 'overall_score'], ascending=False)

In [ ]:
## add top-N down-modulators per target with their SMILES + logfc.
## Aggregates df_raw at (gene, compound) level (mean logfc across replicates),
## ranks by lowest logfc, keeps top-N, and pivots wide so target_final gets
## columns top1_compound / top1_logfc / top1_smiles … topN_compound / … / topN_smiles.
## SMILES are pulled from serac_df (canonical chem library).
TOP_N = 5

# 1) per-gene compound logfc (mean across replicates) + SMILES from serac_df
agg = (df_raw.groupby(['genes', 'compound'])['logfc'].mean()
       .reset_index()
       .rename(columns={'genes': 'gene'})
       .merge(serac_df[['compound', 'smiles']].drop_duplicates('compound'),
              on='compound', how='left'))

# 2) keep top-N by lowest logfc per gene
agg = agg.sort_values(['gene', 'logfc'], ascending=[True, True])
agg['rank'] = agg.groupby('gene').cumcount() + 1
agg = agg[agg['rank'] <= TOP_N]

# 3) pivot wide → topK_compound / topK_logfc / topK_smiles for K = 1..TOP_N
wide = agg.pivot(index='gene', columns='rank',
                 values=['compound', 'logfc', 'smiles'])
wide.columns = [f'top{int(rk)}_{name}' for name, rk in wide.columns]
# enforce a stable column order so downstream cells can rely on it
wide = wide.reindex(columns=[f'top{k}_{n}' for k in range(1, TOP_N + 1)
                              for n in ('compound', 'logfc', 'smiles')])
wide = wide.reset_index()

# 4) drop any existing top* columns (idempotent re-run) and merge fresh
_top_cols = [c for c in target_final.columns
             if c.split('_', 1)[0].lstrip('top').isdigit() or c in (
                 'top_compound', 'top_logfc', 'top_smiles')]
# tighter filter: only top<int>_{compound,logfc,smiles} or the legacy top_* triple
_top_cols = [c for c in target_final.columns
             if c in {'top_compound', 'top_logfc', 'top_smiles'}
             or any(c == f'top{k}_{n}' for k in range(1, 50)
                                       for n in ('compound', 'logfc', 'smiles'))]
target_final = target_final.drop(columns=_top_cols).merge(wide, on='gene', how='left')

print(f'> annotated {target_final["top1_compound"].notna().sum():,} / {len(target_final):,} '
      f'genes with their top-{TOP_N} down-modulators '
      f'({target_final["top1_smiles"].notna().sum():,} with SMILES from serac_df)')
target_final.head(5)


# ============================================================
# prioritised disease area per gene
# ============================================================
## For each gene, pick the highest-priority therapeutic area among the diseases
## OpenTargets associates with it. PRIORITY is a flagship-pharma ordering (not
## a clinical hierarchy) — uncomment trailing entries for franchises you care
## about (eg ophthalmology, respiratory, ID). Tie-break within the chosen area
## by descending overall_score so the winning disease is the strongest hit there.
PRIORITY = [
    'cancer or benign tumor',                       # BMS Opdivo/Yervoy, Roche Herceptin/Tecentriq, NVS Kisqali/Pluvicto, Pfizer Ibrance/Padcev, Lilly Verzenio/Jaypirca
    'hematologic disease',                          # BMS Revlimid/Pomalyst, Pfizer Elrexfio, Roche Hemlibra
    'cardiovascular disease',                       # BMS Eliquis, NVS Entresto/Leqvio, Pfizer Vyndaqel
    'immune system disease',                        # BMS Zeposia/Sotyktu, NVS Cosentyx/Xolair, Pfizer Xeljanz, Lilly Olumiant
    'musculoskeletal or connective tissue disease', # RA / lupus / psoriasis (overlaps immune)
    'nervous system disease',                       # NVS Kesimpta/Gilenya, Roche Ocrevus/Evrysdi, Lilly Kisunla, BMS Cobenfy
    'psychiatric disorder',                         # BMS Cobenfy, Lilly atypicals
    'nutritional or metabolic disease',             # Lilly Mounjaro/Zepbound; Pfizer & NVS chasing GLP-1
    'endocrine system disease',                     # hormone/metabolic overlap, hormone-driven cancers
    # 'disorder of visual system',                  # Roche Vabysmo/Lucentis
    # 'respiratory or thoracic disease',            # COPD/asthma plays
    # 'infectious disease',                         # Pfizer Comirnaty/Paxlovid/Prevnar
]
_rank = {area: i for i, area in enumerate(PRIORITY)}

# explode therapeutic_areas → one row per (gene, disease, area), filter to areas in PRIORITY
_areas = (ot_df[['target_symbol', 'overall_score', 'disease_name', 'therapeutic_areas']]
          .assign(area=lambda d: d['therapeutic_areas'].fillna('').str.split('|'))
          .explode('area'))
_areas = _areas[_areas['area'].isin(_rank)].copy()
_areas['_rank'] = _areas['area'].map(_rank)

# pick the (area, disease) with smallest rank, break ties by overall_score desc
_top_area = (_areas
             .sort_values(['target_symbol', '_rank', 'overall_score'],
                          ascending=[True, True, False])
             .drop_duplicates('target_symbol', keep='first')
             .rename(columns={'target_symbol':  'gene',
                              'area':           'disease_area',
                              'disease_name':   'disease_area_top',
                              'overall_score':  'disease_area_score'})
             [['gene', 'disease_area', 'disease_area_top', 'disease_area_score']])

target_final = (target_final
                .drop(columns=[c for c in
                               ['disease_area', 'disease_area_top', 'disease_area_score']
                               if c in target_final.columns])
                .merge(_top_area, on='gene', how='left'))

# ============================================================
# pharma-patent override: any gene that's a recent big-pharma patent target
# AND clears the R² > 0.1 predictability bar is relabelled 'pharma'. This
# overrides whatever disease_area OpenTargets assigned so the 3D viz calls
# out genes where (a) we can model SAR and (b) pharma is already chasing them.
# ============================================================
PHARMA_PATENT_CSV = 'data/patent/20260511_pharma_sm.csv'
PHARMA_R2_CUTOFF  = 0.1

_pharma_genes = set(pd.read_csv(PHARMA_PATENT_CSV)['gene'].dropna().unique())
_pharma_mask  = (target_final['gene'].isin(_pharma_genes)
                 & (target_final['R2'] > PHARMA_R2_CUTOFF))
_overridden_prev = target_final.loc[_pharma_mask, 'disease_area'].notna().sum()
target_final.loc[_pharma_mask, 'disease_area'] = 'pharma'

print(f'> {target_final["disease_area"].notna().sum():,} / {len(target_final):,} '
      f'genes annotated with a priority disease area')
print(f'> pharma override: {int(_pharma_mask.sum()):,} genes relabelled to "pharma" '
      f'(in patent list & R² > {PHARMA_R2_CUTOFF}; '
      f'{int(_overridden_prev)} of them previously had a disease area)')
print(target_final['disease_area'].value_counts().to_string())


In [ ]:
## graph of fold enrichment vs disease association
## 3D scatter: R² × overall_score × fold-enrichment, coloured by disease_area.
## Implementation lives in python/functions.py (fn.plot_target_3d). This cell
## just wires the params and writes a sharable standalone HTML to Downloads.
import plotly.io as pio
pio.renderers.default = 'vscode'


## Params for plotting
MUST_INCLUDE     = ['UNC45A','KDM1B']
# always show pharma-patent genes (assigned 'pharma' in cell d3fe884f)
_pharma_show = target_final.loc[target_final['disease_area'] == 'pharma', 'gene'].tolist()
MUST_INCLUDE = sorted(set(MUST_INCLUDE) | set(_pharma_show))
print(f'> must-include set: {len(MUST_INCLUDE)} genes '
      f'(of which {len(_pharma_show)} pharma-tagged)')
EXCLUDE_GENES    = []                 # names to drop because their extreme fold squashes the z-axis
MAX_FOLD_PLOT    = 500                # also drop rows whose fold exceeds this (catches future outliers)
TOP_N_HIGHLIGHT  = 30                 # genes closest to the (R²↑, OS↑, fold↑) corner to highlight
MIN_R2_HIGHLIGHT = 0.12               # noise-floor cutoff; genes below this are NOT highlighted
MIN_OS_AUTO      = 1.5 # 0.60               # auto-highlight any gene with overall_score above this

# Per-disease-area palette for the highlighted points. Order mirrors the
# PRIORITY list in cell d3fe884f. Anything outside this dict (or NaN
# disease_area) falls back to NA_AREA_COLOR.
DISEASE_AREA_COLORS = {
    'pharma':                                       active_c,  # deep navy — patent-backed override
    'cancer or benign tumor':                       '#E63946',  # red
    'hematologic disease':                          '#9D0208',  # blood
    'cardiovascular disease':                       '#FB8500',  # orange
    'immune system disease':                        '#2A9D8F',  # teal
    'musculoskeletal or connective tissue disease': '#264653',  # slate
    'nervous system disease':                       '#5A189A',  # purple
    'psychiatric disorder':                         '#9D4EDD',  # violet
    'nutritional or metabolic disease':             '#F4A261',  # tan
    'endocrine system disease':                     '#E9C46A',  # gold
    'disorder of visual system':                    '#48CAE4',  # cyan
    'respiratory or thoracic disease':              '#A8DADC',  # pale cyan
    'infectious disease':                           '#6A4C93',  # plum
}
NA_AREA_COLOR = '#bbbbbb'
## end Params for plotting

# also write a sharable standalone HTML to Downloads (consistent with the 3D t-SNE plot)
DOWNLOADS = '/mnt/c/Users/gtamo/Downloads'

fig, highlighted = fn.plot_target_3d(
    target_final,
    must_include=MUST_INCLUDE,
    exclude_genes=EXCLUDE_GENES,
    max_fold_plot=MAX_FOLD_PLOT,
    top_n_highlight=TOP_N_HIGHLIGHT,
    min_r2_highlight=MIN_R2_HIGHLIGHT,
    min_os_auto=MIN_OS_AUTO,
    disease_area_colors=DISEASE_AREA_COLORS,
    na_area_color=NA_AREA_COLOR,
    html_path=os.path.join(DOWNLOADS, '20260505_R2_vs_disease_vs_fold.html'),
)


In [ ]:
## uniquecontrast extraction for highlighted (gene, top-K compound) pairs.
## For every (gene, top1..topK compound) in `highlighted`, look up the matching
## `uniquecontrast` rows in df_raw. One (gene, compound) pair maps to N rows
## (N = number of plate × replicate measurements), so the result is typically
## several hundred rows for ~20 highlighted genes — NOT exactly N_genes × K.

TOP_N_EXTRACT = 5

# 1) reshape highlighted's top1..topN columns to long format: (gene, rank, compound)
records = []
for _, row in highlighted.iterrows():
    for k in range(1, TOP_N_EXTRACT + 1):
        cmp = row.get(f'top{k}_compound')
        if pd.notna(cmp):
            records.append({'gene': row['gene'], 'rank': k, 'compound': str(cmp)})
gene_compound = pd.DataFrame(records)
print(f'> {len(gene_compound):,} (gene, top-K compound) pairs from {highlighted.shape[0]} highlighted genes')

# 2) inner-join to df_raw on (gene, compound)
highlighted_uniquecontrasts = (
    gene_compound
    .merge(df_raw[['genes', 'compound', 'MSPlate', 'uniquecontrast', 'logfc']],
           left_on=['gene', 'compound'], right_on=['genes', 'compound'],
           how='inner')
    .drop(columns='genes')
    [['compound', 'gene', 'rank', 'MSPlate', 'uniquecontrast', 'logfc']]
    .drop_duplicates()
    .sort_values(['gene', 'rank'])
    .reset_index(drop=True)
)

print(f'> {len(highlighted_uniquecontrasts):,} (compound, gene, uniquecontrast) rows '
      f'across {highlighted_uniquecontrasts["gene"].nunique()} genes  '
      f'(~{len(highlighted_uniquecontrasts) / max(1, highlighted_uniquecontrasts["gene"].nunique()):.1f} per gene)')

# any (gene, compound) pairs that produced zero uniquecontrast matches?
matched_pairs = set(zip(highlighted_uniquecontrasts['gene'],
                        highlighted_uniquecontrasts['compound']))
unmatched = [(g, c) for g, c in zip(gene_compound['gene'], gene_compound['compound'])
             if (g, c) not in matched_pairs]
if unmatched:
    print(f'> [warn] {len(unmatched)} (gene, compound) pairs had no match in df_raw '
          f'(e.g. {unmatched[:3]})')

highlighted_uniquecontrasts.head(10)
out = '/mnt/c/Users/gtamo/Serac Biosciences Dropbox/Serac_team/12_Proteomics/5_Inventory/CDDVault/Proteomics/GiorgioTamo/20260508_ML_selected_genes.csv' 
highlighted_uniquecontrasts.to_csv(out,sep=',',index=False)

In [ ]:
## Maximum Common Substructure (MCS) of the top-K most active compounds vs the rest.
## Sweeps K to see how the signal evolves with stringency:
##   - low K (5)   → cleanest MCS but few datapoints; can be unstable
##   - mid K (10)  → usually the sweet spot
##   - high K (20) → tests whether the signal generalises beyond the very tail
## For each K, we report the MCS size + the 2×2 enrichment metrics so you can pick
## the K where the scaffold is both *large enough to be informative* and *still
## significantly enriched*. The MCS for the most-enriched K is rendered at the bottom.
from rdkit.Chem import rdFMCS, Draw
from scipy.stats import fisher_exact

## Compute enrichment for specific genes
gene = 'ACADVL'

tardf = df_raw[df_raw['genes'] == gene]

# mean logfc per compound for this gene
tardf_ml          = tardf.groupby('compound')['logfc'].mean().reset_index()
tardf_ml['label'] = tardf_ml['logfc']
tardf_ml          = tardf_ml.dropna()
tardf_ml          = pd.merge(serac_df,tardf_ml).drop_duplicates()


K_VALUES      = [3, 5, 10, 20]
MCS_THRESHOLD = 0.7        # MCS must appear in ≥X% of actives
MCS_TIMEOUT   = 30         # seconds, per K
DRAW_BY       = 'fold'     # how to RANK Ks: 'fold' (chemistry-meaningful effect size)
                           # or 'fisher_p' (rewards sample size — picks bigger K).
DRAW_TOP_N    = 3          # how many of the top-ranked Ks to render side-by-side

# rank compounds by mean logfc (most-negative first → most active for degraders)
df_g = tardf_ml.copy().sort_values('logfc', ascending=True).reset_index(drop=True)
n_total = len(df_g)
print(f'> gene={gene}  |  {n_total:,} compounds with mean logfc; sweeping K = {K_VALUES}\n')

records = []
mcs_by_K = {}                                              # cache for the optional draw
for K in K_VALUES:
    if K > n_total:
        records.append({'K': K, 'note': f'only {n_total} compounds available'})
        continue

    top_smiles  = df_g['smiles'].iloc[:K].tolist()
    rest_smiles = df_g['smiles'].iloc[K:].tolist()
    mols_top  = [m for m in (Chem.MolFromSmiles(s) for s in top_smiles)  if m]
    mols_rest = [m for m in (Chem.MolFromSmiles(s) for s in rest_smiles) if m]

    mcs = rdFMCS.FindMCS(
        mols_top,
        threshold=MCS_THRESHOLD,
        bondCompare=rdFMCS.BondCompare.CompareOrderExact,
        atomCompare=rdFMCS.AtomCompare.CompareElements,
        ringMatchesRingOnly=True, completeRingsOnly=True,
        timeout=MCS_TIMEOUT,
    )
    pat = Chem.MolFromSmarts(mcs.smartsString) if mcs.numAtoms > 0 else None
    mcs_by_K[K] = (mcs, pat, mols_top, mols_rest)

    if pat is None or mcs.numAtoms == 0:
        records.append({'K': K, 'mcs_atoms': 0, 'note': 'no MCS found'})
        continue

    a = sum(1 for m in mols_top  if m.HasSubstructMatch(pat))
    b = len(mols_top)  - a
    c = sum(1 for m in mols_rest if m.HasSubstructMatch(pat))
    d = len(mols_rest) - c

    fold = (a / max(a + b, 1)) / (max(c, 1) / max(c + d, 1))
    odds = ((a + 0.5) * (d + 0.5)) / ((b + 0.5) * (c + 0.5))
    _, p_fisher = fisher_exact([[a, b], [c, d]], alternative='greater')

    records.append({
        'K':           K,
        'logfc_max':   round(df_g['logfc'].iloc[K-1], 3),    # weakest active in this top-K
        'mcs_atoms':   mcs.numAtoms,
        'in_top':      f'{a}/{a+b}',
        'in_top_pct':  round(a / max(a+b,1) * 100, 1),
        'in_rest':     f'{c}/{c+d}',
        'in_rest_pct': round(c / max(c+d,1) * 100, 2),
        'fold':        round(fold, 1) if c > 0 else float('inf'),
        'odds_ratio':  round(odds, 1),
        'fisher_p':    p_fisher,
    })

res_df = pd.DataFrame(records)
print(res_df.to_string(index=False))

# pick the most-enriched K and draw its MCS
ranked = res_df.dropna(subset=['fisher_p']) if 'fisher_p' in res_df.columns else res_df
if len(ranked):
    if DRAW_BY == 'fisher_p':
        ranked = ranked.sort_values(['fisher_p', 'mcs_atoms'], ascending=[True, False])
    elif DRAW_BY == 'fold':
        ranked = ranked.sort_values(['fold', 'mcs_atoms'], ascending=[False, False])
    else:
        raise ValueError("DRAW_BY must be 'fisher_p' or 'fold'")
    # render the top-N most-enriched Ks side-by-side. Default ranks by `fold` so
    # the strongest chemistry-signal MCS wins (vs `fisher_p` which favours large K).
    for rank, (_, row) in enumerate(ranked.head(DRAW_TOP_N).iterrows(), start=1):
        K = int(row['K'])
        if K not in mcs_by_K or mcs_by_K[K][1] is None:
            continue
        print(f'\n────  rank {rank}  ·  K={K}  ────')
        print(f'  fold={row["fold"]}×  ·  fisher_p={row["fisher_p"]:.2e}  ·  '
              f'in_top={row["in_top_pct"]}%  ·  in_rest={row["in_rest_pct"]}%  ·  '
              f'mcs={row["mcs_atoms"]} atoms')
        display(Draw.MolToImage(mcs_by_K[K][1], size=(450, 300)))

        top_compounds_K = (df_g.head(K)[['compound', 'smiles', 'logfc']]
                                .reset_index(drop=True))
        print(f'  Top-{K} compounds (most-negative logfc first):')
        display(top_compounds_K)

    # `top_compounds` keeps the rank-1 set so the downstream Draw-grid cell still works
    best_row = ranked.iloc[0]
    best_K   = int(best_row['K'])
    top_compounds = (df_g.head(best_K)[['compound', 'smiles', 'logfc']]
                          .reset_index(drop=True))
else:
    print('\n→ no MCS found at any K — try lowering MCS_THRESHOLD or increasing K_VALUES')

In [ ]:
## Draw top compounds:
## Renders the top-K compounds (already pulled into `top_compounds` by the cell
## above) as a labelled grid. Each tile shows the compound id + logfc value.
rdkit_tools.draw_compounds_grid(
    top_compounds,
    compound_col='compound',
    smiles_col='smiles',
    extra_cols=['logfc'],
    mols_per_row=5,
    sub_img_size=(280, 220),
)

In [ ]:
## Bemis-Murcko scaffold enrichment for the top-K most-active compounds
## against the current gene. "Top-K" = K compounds with the most negative
## logfc (= strongest down-regulation). Each scaffold is tested with a
## one-sided Fisher exact ("over-represented in top-K vs rest"); sort by
## p-value and show the top 15 scaffolds.
from rdkit.Chem.Scaffolds import MurckoScaffold
from scipy.stats import fisher_exact

TOP_K = 10
df_g = tardf_ml.copy()
# rank by logfc ascending (most negative first); top-K = active, rest = control
df_g = df_g.sort_values('logfc', ascending=True).reset_index(drop=True)
df_g['active'] = 0
df_g.loc[:TOP_K - 1, 'active'] = 1
n_top  = int((df_g['active'] == 1).sum())
n_rest = int((df_g['active'] == 0).sum())
cutoff_logfc = df_g.loc[TOP_K - 1, 'logfc'] if n_top else float('nan')
print(f'> gene={gene}  |  top-{n_top} (logfc ≤ {cutoff_logfc:.3f}) vs {n_rest} rest, total {len(df_g):,}')

if n_top == 0:
    print(f'> not enough compounds to take top-{TOP_K} — only {len(df_g)} available')
else:
    def _scaffold_smiles(smi):
        m = Chem.MolFromSmiles(smi) if isinstance(smi, str) else None
        return Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(m)) if m else None

    df_g['scaffold'] = df_g['smiles'].apply(_scaffold_smiles)
    # drop rows where scaffold extraction failed or returned empty (acyclic compounds)
    valid = df_g.dropna(subset=['scaffold'])
    valid = valid[valid['scaffold'] != '']

    # contingency: scaffold present (top vs rest) × active vs not
    counts = (valid.groupby(['scaffold', 'active']).size().unstack(fill_value=0)
                  .rename(columns={0: 'rest', 1: 'top'}))
    # fisher exact, alternative='greater' → over-representation in top-K
    counts['fisher_p'] = counts.apply(
        lambda r: fisher_exact(
            [[r['top'], n_top  - r['top']],
             [r['rest'], n_rest - r['rest']]],
            alternative='greater',
        ).pvalue, axis=1)
    # enrichment = (frac in top-K) / (frac in rest); +1 in denom to avoid div/0
    counts['enrichment'] = (counts['top'] / n_top) / ((counts['rest'] + 1) / n_rest)
    counts = counts.sort_values('fisher_p').head(15)
    print(counts.to_string())